# HyPRNN demo pipeline

An end-to-end, **fast** walkthrough of the whole project on a single microstructure:

1. **YADE deposition** — settle a small filler RVE (GUI shown).
2. **Data from mesh** — mesh the packing and run a few RVE loading paths.
3. **Train a HyPRNN** — on a *precomputed* dataset (a few minutes on CPU).
4. **Optimize a hole-bulge design** — CMA-ES with the trained surrogate.
5. **Visualize** the results.

The heavy steps are deliberately shrunk (≈8 RVE samples, coarse macro mesh, few
CMA-ES iterations) so nothing runs for hours — tweak the `# knob` variables to
scale up. Everything runs on CPU.

**Prerequisites**
- Run inside the `hyprnn_env` conda env (`jupyterlab` + `ipykernel` are in `environment.yml`).
- **Step 1 needs a system YADE install** (`yadedaily`/`yade`); if absent, that cell
  falls back to an existing packing under `yade/data/`.
- Step 4's plotting uses matplotlib mathtext (we disable `usetex`), so no LaTeX
  install is required.

This notebook only *calls* the project's modules — it does not modify any script.

## 0 · Setup — paths and imports

In [ ]:
import sys, os
from pathlib import Path

# Find the repo root = the folder that contains material_params.py.
REPO = Path.cwd()
while not (REPO / "material_params.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the repository.")
    REPO = REPO.parent

# Put the repo root and each script folder on the import path, so both
# package-style (scripts_materials.x) and folder-style (from data_utils import x)
# imports used across the project resolve.
for p in [REPO, REPO/"scripts_surrogates", REPO/"scripts_FEM",
          REPO/"scripts_materials", REPO/"scripts_data_creation", REPO/"yade"]:
    sys.path.insert(0, str(p))

os.environ["JAX_PLATFORMS"] = "cpu"        # keep the demo light and deterministic

DEMO = REPO/"results"/"demo"               # all demo outputs live here
DEMO.mkdir(parents=True, exist_ok=True)
print("Repo root :", REPO)
print("Demo dir  :", DEMO)

## 1 · YADE deposition

Deposits a **small** packing of wood chips + filler and slices it into a periodic
2D RVE. The YADE GUI opens so you can watch it settle; **close the window when it
finishes** to hand control back to the notebook (it auto-saves the packing).

> The only step needing an external YADE install. Set `YADE_BIN` to override the
> runner (e.g. `YADE_BIN=yade`). Tweak `--num_chips` / `--filler_fraction` freely.

In [ ]:
import subprocess

YADE = os.environ.get("YADE_BIN", "yadedaily")
PACKING = DEMO/"packing_2D.npy"

cmd = [YADE, "deposit_rve.py", "--",
       "--seed", "0",
       "--num_chips", "120",          # knob: small = fast to settle
       "--filler_fraction", "0.5",
       "--output", str(PACKING)]      # 2D packing is written here (+ _meta, coords_3D)

try:
    print("Launching:", " ".join(cmd), "\n(close the YADE window when it finishes)")
    subprocess.run(cmd, cwd=str(REPO/"yade"), check=True)
    print("Saved packing ->", PACKING)
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    print(f"YADE step skipped ({e}). Step 2 will fall back to an existing packing.")

## 2 · Data from a mesh

Mesh the packing (wood → inclusions, filler stripped → matrix) and run a handful
of RVE homogenizations, each along a random loading direction. This is a
miniature of `scripts_data_creation/create_deposition_data.py` (8 samples instead
of thousands) and produces `{F, PK1}` sequences plus a `matparam` table.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = False
from packing_to_mesh import mesh_file
from scripts_materials.rve_material import RVEMaterial
from material_params import WOOD_E, WOOD_NU, FUNGI_MU, FUNGI_LAMBDA

# Use the fresh packing if present, else an existing one from the yade dataset.
packing = PACKING if PACKING.exists() else next((REPO/"yade"/"data").rglob("coords_2D_*.npy"))
print("Meshing:", packing)
msh = mesh_file(str(packing), output=str(DEMO/"rve_demo.msh"), shrink_factor=0.85)

# ---- knobs ----
N_SAMPLES = 8
TIMESTEPS = 50
DISP_INCR = 0.01
rng = np.random.default_rng(0)

mat_props = {"wood":  {"E": WOOD_E, "nu": WOOD_NU, "tag": 2},
             "fungi": {"mu": FUNGI_MU, "lambda_": FUNGI_LAMBDA, "tag": 1}}

def random_F_dir():
    while True:
        d = rng.normal(size=3) * 0.5
        F = np.eye(2) + np.array([[d[0], d[1]], [d[1], d[2]]])
        if 0.2 < np.linalg.det(F) < 5.0:
            return F - np.eye(2)

F_all   = np.zeros((N_SAMPLES, TIMESTEPS, 2, 2))
PK1_all = np.zeros((N_SAMPLES, TIMESTEPS, 2, 2))
for s in range(N_SAMPLES):
    rve = RVEMaterial(mesh_file=str(msh), material_properties=mat_props, visualize=False)
    Fdir, converged = random_F_dir(), True
    for t in range(TIMESTEPS):
        F = np.eye(2) + (t + 1) * DISP_INCR * Fdir
        PK1, cauchy, converged = rve.update_stress(F, monitor_base_solve=False)
        F_all[s, t], PK1_all[s, t] = F, PK1
    print(f"  sample {s}: {'ok' if converged else 'last step not converged'}")

# Save in the same layout the training scripts expect.
np.save(DEMO/"demo_F.npy", F_all)
np.save(DEMO/"demo_PK1.npy", PK1_all)

fig, ax = plt.subplots(figsize=(4, 3))
for s in range(N_SAMPLES):
    ax.plot(F_all[s, :, 0, 0], PK1_all[s, :, 0, 0], marker=".")
ax.set_xlabel(r"$F_{xx}$"); ax.set_ylabel(r"$P_{xx}$")
ax.set_title(f"{N_SAMPLES} RVE loading paths"); plt.show()

## 3 · Train a HyPRNN

Training on the ≈8 samples above would not learn anything, so — as intended — we
train on a **precomputed** deposition-filler dataset. The config mirrors
`train_surrogate_deposit_batch.py` exactly (so the saved model loads unchanged in
step 4); only `max_epochs`/`patience` are trimmed for a few-minute run.

> The learning curve is plotted **after** training. A *live* curve would need a
> per-epoch callback inside `trainer.py` — out of scope here (would require a
> script change).

In [ ]:
import jax
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["text.usetex"] = False
from scripts_surrogates.data_utils import LDDataset, Config, save_settings
from scripts_surrogates.HyPRNN import create_shared_hyper_prnn_model
from scripts_surrogates.trainer import Trainer

DATA = REPO/"data"/"deposition"/"filler"/"dataset_combi_v4v6"/"mixed_t50_merged"  # knob: point at any dataset
assert Path(str(DATA) + "_F.npy").exists(), f"Dataset not found: {DATA}_F.npy"

cfg = dict(
    seq_length=50, num_samples=512,
    train_samples=256, train_batch_size=2, val_test_samples=128,
    model_type="hyprnn", encoder_type="Linear", decoder_type="HyperSparseLayer",
    mat_parameters=["mu", "lambda", "fil_frac"], norm_matparams=[False, False, True],
    shared_micro_features=[2], mat_micro_features=[0, 1],
    hyper_hidden_sizes=(8,), hyper_activation="sigmoid",
    mat_points=6, norm_stresses=True, stress_scaling_feature=None,
    lr_schedule_steps=10000, warmup_epochs=0, base_lr=1e-3, min_lr_factor=1,
    max_epochs=300, patience=40, interval=1, verbose=False, seed=1,   # knob: raise for accuracy
)

dataset = LDDataset(str(DATA), seq_length=cfg["seq_length"], num_samples=cfg["num_samples"],
                    mat_file=str(DATA) + "_matparam.data", mat_features=cfg["mat_parameters"],
                    norm_stresses=cfg["norm_stresses"], norm_matparams=cfg["norm_matparams"],
                    stress_scaling_feature=cfg["stress_scaling_feature"])

idx = np.arange(cfg["num_samples"])
train_pool = idx[:-2 * cfg["val_test_samples"]]
val_idx    = idx[-2 * cfg["val_test_samples"]:-cfg["val_test_samples"]]
test_idx   = idx[-cfg["val_test_samples"]:]
np.random.seed(cfg["seed"])
sel = np.random.permutation(train_pool)[:cfg["train_samples"]]
trainset = dataset.get_subset(sel)
valset   = dataset.get_subset(val_idx)
testset  = dataset.get_subset(test_idx)

key = jax.random.PRNGKey(cfg["seed"])
model, params, material = create_shared_hyper_prnn_model(
    random_key=key, n_micro_raw=len(cfg["mat_parameters"]),
    shared_micro_features=cfg["shared_micro_features"], n_matpts=cfg["mat_points"],
    encoder_type=cfg["encoder_type"], stress_normalizer=dataset.stress_normalizer,
    mat_m_feats=cfg["mat_micro_features"], hyper_hidden_sizes=cfg["hyper_hidden_sizes"],
    hyper_activation=cfg["hyper_activation"], stress_scaling_index=None)

lr_config = Config(warmup_epochs=cfg["warmup_epochs"], schedule_steps=cfg["lr_schedule_steps"],
                   steps_per_epoch=cfg["train_samples"] / cfg["train_batch_size"],
                   base_learning_rate=cfg["base_lr"], min_lr_factor=cfg["min_lr_factor"])
trainer = Trainer(model, params, material=material, lr_config=lr_config,
                  random_key=key, out_norm_factor=1)
trainer.train(trainset, valset, test_data=testset, **cfg)

# Save with the exact {loc}.npy / {loc}_settings.json / {loc}_normparams.json layout.
MODEL = DEMO/"hyprnn_demo"
trainer.save(str(MODEL))
dataset.saveDataparams(str(MODEL) + "_normparams")
save_settings(cfg, str(MODEL) + "_settings")
print("Saved model ->", MODEL)

tr, vl, te = trainer.get_losses()
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(tr, label="train"); ax.plot(vl, label="val"); ax.plot(te, label="test")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE"); ax.legend()
ax.set_title("Learning curve"); plt.show()

## 4 · Optimize a hole-bulge design

A plate with a central hole is compressed between rigid platens; every element is
the HyPRNN surrogate trained above, parametrized by a per-hex filler fraction and
fiber orientation. CMA-ES searches the grading to **maximize the hole bulge**.

Kept cheap: coarse macro mesh, large hex cells (few regions), a handful of CMA-ES
iterations. Each evaluation is one nonlinear macro FE solve.

In [ ]:
import matplotlib.pyplot as plt
from hole_deformation_optimization import BulgeOptimizer
plt.rcParams["text.usetex"] = False   # neutralize the module's usetex=True (no LaTeX needed)

opt_out = DEMO/"bulge_opt"
optimizer = BulgeOptimizer(
    output_folder=str(opt_out),
    prnn_model_loc=str(MODEL),         # the model trained in step 3
    macro_meshsize=0.06,               # knob: coarse = fast
    hole_radius=0.25, plate_height=1.0,
    hex_size=0.25,                     # knob: large = few design regions
    symmetric_h=True, symmetric_v=True,
    fil_bounds=(0.0, 1.0), max_disp=0.15,
)
result = optimizer.run_cmaes(sigma0=0.3, maxiter=4, seed=1, popsize=6)   # knobs: maxiter / popsize
print("Best |bulge|:", getattr(result, "fbest", result))

## 5 · Visualize results

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
plt.rcParams["text.usetex"] = False

# CMA-ES progress (objective per evaluation).
hist = np.asarray(optimizer.history)
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(hist, ".-")
ax.set_xlabel("CMA-ES evaluation"); ax.set_ylabel("|bulge|")
ax.set_title("Optimization progress"); plt.show()

# Field/grading figures the optimizer wrote to disk.
pngs = sorted(glob.glob(str(opt_out/"*.png")))
print(f"{len(pngs)} PNG figures in {opt_out}")
for p in pngs[:6]:
    display(Image(filename=p))
pdfs = sorted(glob.glob(str(opt_out/"*.pdf")))
if pdfs:
    print("PDF outputs:", *pdfs, sep="\n  ")